#利用 EfficientNetB1 進行口罩是否佩戴正確的多元分類任務，使用PyTorch實踐
參考自[EfficientNet Implementation from scratch in Pytorch: A step-by-step guide](https://medium.com/@aniketthomas27/efficientnet-implementation-from-scratch-in-pytorch-a-step-by-step-guide-a7bb96f2bdaa)

#[訓練好的模型在這](https://huggingface.co/spaces/JasonHoooooo/Face_Mask_Classification?logs=container)(可能需要重新restart this space)


In [ ]:
import gdown
import zipfile
import os # 主要用於文件管理
import shutil # 提供了更高級的文件操作功能，尤其是在文件的複製、移動、刪除方面
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from pathlib import Path
from PIL import Image
from torchvision import transforms
from torch.utils.data import DataLoader
import torch
from torch import nn
from math import ceil
!pip install torchinfo
from torchinfo import summary
from google.colab import drive
!pip install gradio
import gradio as gr
import numpy as np
from tqdm.auto import tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 10.4 MB/s eta 0:00:00


In [ ]:
# 下載檔案

gdown.download("https://drive.google.com/uc?export=download&id=1-M7ki8zY2c-e0_lt1T80ALvbhHvefp2-", "Face_Mask_Dataset.zip", quiet=False)
# 原始的分享連結不行，要使用Google Drive Direct Link Generator轉換
# 再加上gdown（專門用在Google Drive文件下載的庫），因為用requests的話，
# 大的檔案會有google網頁下載提示，這樣下載的東西會變成html網頁
# 連結載入太多次不能用的時候就再上傳一次zip檔到雲端，用新連結

Downloading...
From (original): https://drive.google.com/uc?export=download&id=1-M7ki8zY2c-e0_lt1T80ALvbhHvefp2-
From (redirected): https://drive.google.com/uc?export=download&id=1-M7ki8zY2c-e0_lt1T80ALvbhHvefp2-&confirm=t&uuid=3ca8cf96-96b6-4f0f-be5b-cc47f09cf5a8
To: /content/Face_Mask_Dataset.zip
100%|██████████| 2.51G/2.51G [00:36<00:00, 68.4MB/s]


'Face_Mask_Dataset.zip'

In [ ]:
# 解壓縮zip並忽略__MACOSX資料夾

with zipfile.ZipFile("Face_Mask_Dataset.zip") as zip_file:
    for file in zip_file.namelist():
        if not file.startswith("__MACOSX"):
            zip_file.extract(file, "Face_Mask_Dataset")

In [ ]:
# 檔案原本長這樣：Face_Mask_Dataset/Face_Mask_Dataset/4個戴口罩的mask_class，所以要刪除多餘的資料夾

# dir=directory=目錄，就是資料夾名稱
inner_dir_path = os.path.join("Face_Mask_Dataset", "Face_Mask_Dataset") # 把輸入的string組成path

for mask_class in os.listdir(inner_dir_path):
    src_path = os.path.join(inner_dir_path, mask_class) # src=source
    dst_path = os.path.join("Face_Mask_Dataset", mask_class) # dst=destination
    shutil.move(src_path, dst_path) # 從src_path移動到dst_path

os.rmdir(inner_dir_path) # 刪除空的Face_Mask_Dataset/Face_Mask_Dataset資料夾

In [ ]:
# 刪除非空的檔案，需要時再用
# path = "Face_Mask_Dataset"
# shutil.rmtree(path)

In [ ]:
# 分成train data, test data，並刪除原本的4個mask_class與.DS_Store

for dirpath, dirnames, filenames in os.walk("Face_Mask_Dataset"): # 資料夾名稱 子目錄名稱 子目錄下的檔案名稱

    if not dirnames: # 跟下第二個註解有關
        continue

    for mask_class in dirnames: # 這個迴圈只需要在walk"Face_Mask_Dataset"這個路徑用
        original_mask_class_path = os.path.join(dirpath, mask_class) # dirpath="Face_Mask_Dataset"
        train_path = os.path.join("Face_Mask_Dataset", "train", mask_class)
        test_path = os.path.join("Face_Mask_Dataset", "test", mask_class)

        os.makedirs(train_path, exist_ok=True) # 依照給的路徑建立資料夾(dir)
        os.makedirs(test_path, exist_ok=True)

        files = [f for f in os.listdir(original_mask_class_path)]

        train_files, test_files = train_test_split(files, test_size=0.2, random_state=1)

        for file in train_files:
            src_path = os.path.join(original_mask_class_path, file)
            dst_path = os.path.join(train_path, file)
            shutil.move(src_path, dst_path)

        for file in test_files:
            src_path = os.path.join(original_mask_class_path, file)
            dst_path = os.path.join(test_path, file)
            shutil.move(src_path, dst_path)

        os.rmdir(original_mask_class_path)

for root, dirs, files in os.walk("Face_Mask_Dataset"): # 不與上面的loop合併是因為上面在做檔案處理，路徑會一直變
    for file in files:
        if file == '.DS_Store':
            file_path = os.path.join(root, file)
            if os.path.exists(file_path):
                os.remove(file_path)
                print(f"Deleted: {file_path}")

Deleted: Face_Mask_Dataset/.DS_Store
Deleted: Face_Mask_Dataset/test/without_mask/.DS_Store
Deleted: Face_Mask_Dataset/test/with_mask/.DS_Store


In [ ]:
# 建立ImageDataset

class ImageDataset(Dataset):
  def __init__(self, dataset_root, train, transform=None): #不繼承dataset也可以用？！

    if train:
      train_or_test_path = Path(dataset_root) / "train"
    else:
      train_or_test_path = Path(dataset_root) / "test"

    self.classes = []
    for class_name in sorted(os.listdir(train_or_test_path)): # listdir是無序的，用sorted按照字母排序
        self.classes.append(class_name)
    self.paths = [f for f in train_or_test_path.rglob("*") if f.is_file()] #image_path.rglob("*")讀取此根下所有檔案與資料夾
    self.transform = transform #將參數指派給屬性

  def __getitem__(self, index): #__getitem__是py內建的特殊方法，可以利用[]呼叫item
    img = Image.open(self.paths[index]).convert("RGB")
    class_name = self.paths[index].parent.name
    class_idx = self.classes.index(class_name)

    if self.transform:
      return self.transform(img), class_idx
    else:
      return img, class_idx

  def __len__(self): #__len__是py內建的特殊方法，可以利用len()取得長度
    return len(self.paths)

In [ ]:
train_tranform = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor()
])

test_tranform = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.ToTensor()
])

In [ ]:
train_dataset = ImageDataset("Face_Mask_Dataset", train=True, transform=train_tranform)
test_dataset = ImageDataset("Face_Mask_Dataset", train=False, transform=test_tranform)

# train_dataset.classes = ['without_mask', 'incorrect_mask_mmc', 'incorrect_mask_mc', 'with_mask']
# len(train_dataset), len(test_dataset) = (11630, 2908)

# x, y = train_dataset[9999]
# import matplotlib.pyplot as plt
# plt.imshow(x.permute(1, 2, 0))

In [ ]:
torch.manual_seed(87)

BATCH_SIZE = 32

train_dataloader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

x_first_batch, y_first_batch = next(iter(train_dataloader))
# ptint(x_first_batch.shape, y_first_batch.shape, x_first_batch[0], y_first_batch[0])

# import matplotlib.pyplot as plt
# plt.imshow(x_first_batch[31].permute(1, 2, 0))
# print(y_first_batch[31])

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu" #()記得
print(device)

cuda


In [ ]:
mb_params = [
    # expand_ratio, channels, repeated_times, stride, kernel_size
    [1, 16, 1, 1, 3],
    [6, 24, 2, 2, 3],
    [6, 40, 2, 2, 5],
    [6, 80, 3, 2, 3],
    [6, 112, 3, 1, 5],
    [6, 192, 4, 2, 5],
    [6, 320, 1, 1, 3],
]

alpha, beta = 1.2, 1.1

scale_values = {
    # (Φ=phi, resolution, dropout_rate)
    "B0": (0, 224, 0.2),
    "B1": (0.5, 240, 0.2),
    "B2": (1, 260, 0.3),
    "B3": (2, 300, 0.3),
    "B4": (3, 380, 0.4),
    "B5": (4, 456, 0.4),
    "B6": (5, 528, 0.5),
    "B7": (6, 600, 0.5),
}

In [ ]:
class ConvBnSiLUBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding, groups=1):
        super(ConvBnSiLUBlock, self).__init__()
        self.block = nn.Sequential(
                        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, groups=groups), # groups: 分成幾組做Depthwise Convolution，預設是1（不做）
                        nn.BatchNorm2d(out_channels), # 標準化，可緩解gradient消失or爆炸
                        nn.SiLU()
                    )

    def forward(self, x):
        return self.block(x)

In [ ]:
class MBConvBlock(nn.Module): # MBConv=Mobile Inverted Bottleneck Convolution
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding, expand_ratio, reduction=2):
        super(MBConvBlock, self).__init__()
        # self.use_residual = in_channels == out_channels and stride == 1 # Skip Connection
        expanded_channels = in_channels * expand_ratio
        self.is_expand = expand_ratio > 1
        reduced_channels = int(in_channels / reduction) # SE block會用到

        if self.is_expand:
            self.expand_conv = ConvBnSiLUBlock(in_channels, expanded_channels, kernel_size=1, stride=1, padding=0) # Expand Layer

        self.conv = nn.Sequential(
            ConvBnSiLUBlock(expanded_channels, expanded_channels, kernel_size, stride, padding, groups=expanded_channels), # Depthwise Convolution
            SEBlock(expanded_channels, reduced_channels), # Squeeze-and-Excitation Block
            nn.Conv2d(expanded_channels, out_channels, kernel_size=1, stride=1, padding=0), # Pointwise Convolution，因為kernel size是1，所以padding=0
            nn.BatchNorm2d(out_channels),
        )

        # nn.Conv2d的parameters：

        # in_channels (int) – Number of channels in the input image
        # out_channels (int) – Number of channels produced by the convolution
        # kernel_size (int or tuple) – Size of the convolving kernel
        # stride (int or tuple, optional) – Stride of the convolution. Default: 1
        # padding (int, tuple or str, optional) – Padding added to all four sides of the input. Default: 0
        # padding_mode (str, optional) – 'zeros', 'reflect', 'replicate' or 'circular'. Default: 'zeros'
        # dilation (int or tuple, optional) – Spacing between kernel elements. Default: 1
        # groups (int, optional) – Number of blocked connections from input channels to output channels. Default: 1
        # bias (bool, optional) – If True, adds a learnable bias to the output. Default: True

    def forward(self, x):
        if self.is_expand:
            return self.conv(self.expand_conv(x))
        else:
            return self.conv(x)

In [ ]:
class SEBlock(nn.Module): # SE=Squeeze and Excitation
    def __init__(self, in_channels, reduced_channels):
        super(SEBlock, self).__init__()
        self.se = nn.Sequential(
            # Squeeze
            nn.AdaptiveAvgPool2d(1), # 1是將每個channel濃縮（全局平均池化）成一個值的意思
            # Excitation
            nn.Conv2d(in_channels, reduced_channels, 1), # FC: 將channels減少
            nn.SiLU(), # 彎曲的效果
            nn.Conv2d(reduced_channels, in_channels, 1), # FC: 將channels還原
            nn.Sigmoid(), # 輸出在0~1，這樣在加權的時候就不會變太多
        )

    def forward(self, x):
        # Scale
        return x * self.se(x) # Reweighting: 重新加權通道特徵

In [ ]:
class EfficientNet(nn.Module):
    def __init__(self, model_name, out_classes):
        super(EfficientNet, self).__init__()
        phi, resolution, dropout_rate = scale_values[model_name]
        self.depth_multiplier, self.width_multiplier = alpha**phi, beta**phi
        self.last_channels = ceil(1280 * self.width_multiplier) # 最後一層channel數: 通常在Net的最後，會希望有夠大的channel數，以有效提取特徵
        self.blocks_construction()
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate), #隨機丟棄前一層的部分神經元
            nn.Linear(self.last_channels, out_classes)
        )

    def blocks_construction(self):
        channels = int(32 * self.width_multiplier)
        self.blocks = [ConvBnSiLUBlock(in_channels=3, out_channels=channels, kernel_size=3, stride=2, padding=1)]
        in_channels = channels
        for expand_ratio, channels, repeated_times, stride, kernel_size in mb_params: #stage
            # 確保out_channels是4的倍数，這在一些深度學習框架，能提高計算效率
            out_channels = 4 * ceil(int(channels * self.width_multiplier) / 4)
            num_layers = ceil(repeated_times * self.depth_multiplier)

            for layer in range(num_layers): #repeat stage
                # 在每個stage的第一層通常使用>1的stride，在後續的層中則使用1。有助於降低feature map的空間維度，同時在後續層中保持空間資訊。
                if layer != 0:
                    stride = 1
                self.blocks.append(
                    MBConvBlock(
                        in_channels,
                        out_channels,
                        expand_ratio=expand_ratio,
                        stride=stride,
                        kernel_size=kernel_size,
                        padding=kernel_size // 2 # //取商
                    )
                )
                in_channels = out_channels
        # 使用1x1卷積（pointwise convolution），將in_channels的數量減少到所需的out_channels數。
        self.blocks.append(ConvBnSiLUBlock(in_channels, self.last_channels, kernel_size=1, stride=1, padding=0))
        self.blocks = nn.Sequential(*self.blocks)

    def forward(self, x):
        avgpool= nn.AdaptiveAvgPool2d(1)
        flatten = nn.Flatten()
        return self.classifier(flatten(avgpool(self.blocks(x))))

In [ ]:
train_dataset.classes

['incorrect_mask_mc', 'incorrect_mask_mmc', 'with_mask', 'without_mask']

In [ ]:
test_model = EfficientNet(model_name="B1", out_classes=4)
summary(model=test_model,
        input_size=(32, 3, 240, 240),
        col_names=["input_size", "output_size", "num_params"], #col_names要顯示哪些
        row_settings=["var_names"] #顯示變數名稱
)

Layer (type (var_name))                                 Input Shape               Output Shape              Param #
EfficientNet (EfficientNet)                             [32, 3, 240, 240]         [32, 4]                   --
├─Sequential (blocks)                                   [32, 3, 240, 240]         [32, 1343, 8, 8]          --
│    └─ConvBnSiLUBlock (0)                              [32, 3, 240, 240]         [32, 33, 120, 120]        --
│    │    └─Sequential (block)                          [32, 3, 240, 240]         [32, 33, 120, 120]        990
│    └─MBConvBlock (1)                                  [32, 33, 120, 120]        [32, 16, 120, 120]        --
│    │    └─Sequential (conv)                           [32, 33, 120, 120]        [32, 16, 120, 120]        2,077
│    └─MBConvBlock (2)                                  [32, 16, 120, 120]        [32, 16, 120, 120]        --
│    │    └─Sequential (conv)                           [32, 16, 120, 120]        [32, 16, 120, 120]   

In [ ]:
model = EfficientNet(model_name="B1", out_classes=4)
model.to(device)
# y_pred = model(x_first_batch.to(device))
# y_pred.argmax(dim=1)

EfficientNet(
  (blocks): Sequential(
    (0): ConvBnSiLUBlock(
      (block): Sequential(
        (0): Conv2d(3, 33, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (1): BatchNorm2d(33, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU()
      )
    )
    (1): MBConvBlock(
      (conv): Sequential(
        (0): ConvBnSiLUBlock(
          (block): Sequential(
            (0): Conv2d(33, 33, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=33)
            (1): BatchNorm2d(33, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU()
          )
        )
        (1): SEBlock(
          (se): Sequential(
            (0): AdaptiveAvgPool2d(output_size=1)
            (1): Conv2d(33, 16, kernel_size=(1, 1), stride=(1, 1))
            (2): SiLU()
            (3): Conv2d(16, 33, kernel_size=(1, 1), stride=(1, 1))
            (4): Sigmoid()
          )
        )
        (2): Conv2d(33, 16, kernel_size=(1, 1), str

In [ ]:
cost_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(params=model.parameters(), lr=0.0008)

In [ ]:
def accuracy_fn(y_pred, y_true):
  correct_num = (y_pred==y_true).sum()
  acc = correct_num / len(y_true) * 100
  return acc

In [ ]:
def train_step(dataloader, model, cost_fn, optimizer, accuracy_fn, device):
  train_cost = 0
  train_acc = 0
  for batch, (x, y) in enumerate(dataloader): #enumerate可以多回傳現在是第幾個batch
    x = x.to(device)
    y = y.to(device)
    model.train()

    y_pred = model(x)
    cost = cost_fn(y_pred, y) # y_pred是(32, 4)，y是(32, 1)，他會自己計算loss
    train_cost += cost
    train_acc += accuracy_fn(y_pred.argmax(dim=1), y)

    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

  train_cost /= len(train_dataloader) # 每個batch的cost的平均
  train_acc /= len(train_dataloader) # 每個batch的acc的平均

  print(f"\nTrain Cost: {train_cost:.4f}, Train Acc: {train_acc:.2f}")

In [ ]:
def test_step(dataloader, model, cost_fn, accuracy_fn, device):
  test_cost = 0
  test_acc = 0
  model.eval()
  with torch.inference_mode():
    for (x, y) in dataloader:
      x = x.to(device)
      y = y.to(device)
      test_pred = model(x)
      test_cost += cost_fn(test_pred, y)
      test_acc += accuracy_fn(test_pred.argmax(dim=1), y)

    test_cost /= len(test_dataloader)
    test_acc /= len(test_dataloader)

  print(f"Test Cost: {test_cost:.4f}, Test Acc: {test_acc:.2f}\n")

In [ ]:
# function ConnectButton(){
#     console.log("Connect pushed");
#     document.querySelector("#top-toolbar > colab-connect-button").shadowRoot.querySelector("#connect").click()
# }
# setInterval(ConnectButton,60000);



# clearInterval(intervalId) # 恢復原狀用intervalId換成「11477」

In [ ]:
epochs = 25
for epoch in tqdm(range(epochs)): #tqdm進度條
  print(f"Epoch: {epoch}\n-----------------")

  train_step(train_dataloader, model, cost_fn, optimizer, accuracy_fn, device)

  test_step(test_dataloader, model, cost_fn, accuracy_fn, device)

  0%|          | 0/25 [00:00<?, ?it/s]

Epoch: 0
-----------------


/usr/local/lib/python3.10/dist-packages/PIL/Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



Train Cost: 0.8180, Train Acc: 67.70
Test Cost: 0.6658, Test Acc: 79.93

Epoch: 1
-----------------

Train Cost: 0.5358, Train Acc: 80.15
Test Cost: 0.2425, Test Acc: 92.35

Epoch: 2
-----------------

Train Cost: 0.4459, Train Acc: 83.72
Test Cost: 0.3652, Test Acc: 88.12

Epoch: 3
-----------------

Train Cost: 0.3712, Train Acc: 86.91
Test Cost: 0.1801, Test Acc: 94.29

Epoch: 4
-----------------

Train Cost: 0.3278, Train Acc: 88.04
Test Cost: 0.1978, Test Acc: 94.51

Epoch: 5
-----------------

Train Cost: 0.3193, Train Acc: 88.70
Test Cost: 0.1814, Test Acc: 94.32

Epoch: 6
-----------------

Train Cost: 0.2872, Train Acc: 90.29
Test Cost: 0.1457, Test Acc: 95.31

Epoch: 7
-----------------

Train Cost: 0.2545, Train Acc: 91.39
Test Cost: 0.1936, Test Acc: 94.49

Epoch: 8
-----------------

Train Cost: 0.2602, Train Acc: 91.19
Test Cost: 0.1450, Test Acc: 95.51

Epoch: 9
-----------------

Train Cost: 0.2355, Train Acc: 92.25
Test Cost: 0.1405, Test Acc: 95.55

Epoch: 10
-------

In [ ]:
drive.mount('/content/drive') # 把Google Drive掛載到Colab的文件系統中

os.chdir("/content/drive/My Drive") # 把當前工作目錄改到Google Drive的此路徑，之後的文件操作都會在這執行

os.getcwd() # 獲取當前工作目錄

Mounted at /content/drive


'/content/drive/My Drive'

In [ ]:
torch.save(model.state_dict(), "./trained_model_parameters.pth") # ./表示當前工作目錄

In [ ]:
trained_model = EfficientNet("B1", 4)

trained_model.load_state_dict(torch.load("./trained_model_parameters.pth", map_location=torch.device("cpu"))) # 把加載模型加載到cpu上，若要用gpu就刪掉後半部分code

trained_model = trained_model.to(device)


<ipython-input-29-38b95ea5f8b0>:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  trained_model.load_state_dict(torch.load("./trained_model_parameters.pth", map_location=torc

In [ ]:
input = gr.Image(label="圖片")
output = gr.Label(label="分類結果")

tranformer = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.ToTensor()
])

def face_mask_classification(img):

    if isinstance(img, np.ndarray): # 確認img是否為np.ndarray
        img = Image.fromarray(img) # 將img轉為PIL格式，test_transform需要的格式
    img = tranformer(img)
    img = img.reshape(-1, 3, 240, 240) #傳入model的是batch的形式，所以現在也要給它一個batch

    trained_model.eval()
    with torch.inference_mode():
        y_pred = trained_model(img.to(device))

    y_pred = torch.softmax(y_pred, dim=1) # 轉換成機率分佈

    y_pred = y_pred.reshape(-1) #變成一維
    y_pred_list = y_pred.tolist()
    mask_wearing = ["口罩只戴到下巴", "口罩沒遮住鼻子", "有戴好口罩", "沒戴口罩"]
    y_pred_formatted = {mask_wearing[i]: y_pred_list[i] for i in range(4)} # 轉成字典

    return y_pred_formatted

iface = gr.Interface(fn=face_mask_classification,
                     inputs=input,
                     outputs=output,
                     examples=["/content/drive/My Drive/with_mask_example.jpg",
                               "/content/drive/My Drive/without_mask_example.jpg",
                               "/content/drive/My Drive/mmc_example.jpg",
                               "/content/drive/My Drive/mc_example.jpg"],
                     title="Face Mask Classification",
                     description="上傳你有/沒戴口罩的照片",
                     article="Author : Jason Ho 2024 / 10 / 1 ",
                     theme="soft",
                     submit_btn="開始辨識",
                     clear_btn="清空圖片",
                     example_labels=["有戴好口罩", "沒戴口罩", "口罩沒遮住鼻子", "口罩只戴到下巴"]
        )

iface.launch(share=True)



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://69571c3ee987605181.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
